In [1]:
# Import packages
import os
import numpy as np
from sklearn.model_selection import KFold, train_test_split
import matplotlib.pyplot as plt

In [2]:
DATA_DIR = "../data/"
ALPHABET_FILE = os.path.join(DATA_DIR, "Matrices/alphabet")
BACKGROUND_FILE = os.path.join(DATA_DIR, "Matrices/bg.freq.fmt")
BLOSUM_FILE = os.path.join(DATA_DIR, "Matrices/blosum62.freq_rownorm")

### Helper functions

In [3]:
# Generate alphabet for usage
def generate_alphabet():
    alphabet = np.loadtxt(ALPHABET_FILE, dtype=str)
    return alphabet

In [32]:
# Load background frequencies
def background_frequencies(alphabet):
    background_frequencies = np.loadtxt(BACKGROUND_FILE, dtype=float)
    
    background = list()
    for i in range(0, 20):
        background.append(background_frequencies[i])

    return background

In [5]:
# Read peptide file
def load_peptide_data(file_name):

    peptide_file = os.path.join(DATA_DIR, file_name)
    loaded_peptides = np.loadtxt(peptide_file, dtype=str)

    peptides = []

    for line in loaded_peptides:

        peptides.append(line.upper())

    return peptides

In [6]:
# Convert peptide to collection of numbers
def encode(sequence, alphabet):

    enc = [0] * len(sequence)
    convert_noncanonical_a = {
        "U": "C"
    }

    updated_sequence = sequence[:]

    for idx, a in enumerate(sequence):
        if a in convert_noncanonical_a:
            #print(f"Sequence contained non-canonical amino acid {a}. Now converted to {convert_noncanonical_a[a]}")
            a = convert_noncanonical_a[a]
            updated_sequence = sequence[:idx] + a + sequence[idx+1:]

        a_symbol = alphabet.index(a)
        enc[idx] = a_symbol

    return enc, updated_sequence

In [7]:
# Load Blosum
def blosum_to_emission_propabilities(blosum_file, ref_aa, alphabet):
    ref_aa_index = alphabet.index(ref_aa)
    with open(blosum_file, "r") as f:
        for i,line in enumerate(f):
            if i == ref_aa_index:
                #return line
                return list(map(float, line.strip().split(" ")))


### Model

**Investigating biological states**

- State 1 - positively charged regions, N-region
- State 2 - hydrophobic regions, H-region
- State 3 - cleavage site, C-region

In [24]:
class BaumWelch:
    """
    Initialize θ0

    Repeat:
        E-step:
            compute gamma, xi using θk

        M-step:
            compute θk+1 from gamma, xi

        replace θk with θk+1
    """

    def __init__(self, peptides, states, initial_probability, transition_matrix, emission_probability, alphabet):
        
        self.peptides = peptides
        self.states = states
        self.initial_probability = initial_probability
        self.transition_matrix = transition_matrix
        self.emission_probability = emission_probability
        self.alphabet = alphabet

    ###################################################################################################################
    def alpha_initialize(self, peptide):

        # Initialize
        alpha = np.zeros(shape=(self.states, len(peptide)))

        for i in range(self.states):
            alpha[i][0] = self.initial_probability[i] \
                            * self.emission_probability[i][peptide[0]]

        return alpha

    ###################################################################################################################
    def beta_initialize(self, peptide):

        # Initialize
        beta = np.zeros(shape=(self.states, len(peptide)))

        # Defining beta[T] (final position) as 1
        for i in range(self.states):
            beta[i][-1] = 1

        return beta

    ###################################################################################################################
    def forward_probability(self, peptide, alpha):
    
        # print("emission_probability", emission_probability)
        # print("transition_matrix", transition_matrix)

        T = len(peptide)

        c = np.zeros(T)     # Scaling factor
        c[0] = np.sum(alpha[:,0])

        if c[0] == 0:
            c[0] = 1e-300

        alpha[:,0] /= c[0]  # Scale alpha

        for t in range(1, T):

            for i in range(self.states):
                _sum = 0
                for j in range(self.states):

                    _sum +=  alpha[j][t-1] \
                                * self.transition_matrix[j][i]
                
                # Emission is applied AFTER summing over previous states
                alpha[i][t] = _sum * self.emission_probability[i][peptide[t]]

            c[t] = np.sum(alpha[:, t])  # Sum across all states
            if c[t] == 0:
                c[t] = 1e-300
            
            alpha[:, t] /= c[t]     # Scale all of alpha

        return alpha, c
    
    ###################################################################################################################
    def backward_probability(self, peptide, beta, c):

        T = len(peptide)

        beta[:, -1] /= c[-1]    # Scale final column (1) of beta

        for t in range(T - 2, -1, -1):

            for i in range(self.states):
                _sum = 0
                for j in range(0, self.states):
                    _sum += self.emission_probability[j][peptide[t+1]] \
                                * self.transition_matrix[i][j] \
                                    * beta[j][t+1] 
                
                beta[i][t] = _sum
            
            beta[:, t] /= c[t]  # Scale all of beta

        return beta
    
    ###################################################################################################################
    def calculate_gamma(self, alpha, beta):

        numerator = alpha * beta
        denominator = numerator.sum(axis=0)

        gamma = numerator / denominator

        return gamma
    
    ###################################################################################################################
    def calculate_xi(self, peptide, alpha, beta):

        T = len(peptide)
        xi = np.zeros((self.states, self.states, T-1))

        for t in range(T-1):

            denominator = 0
            for i in range(self.states):
                for j in range(self.states):

                    denominator += alpha[i][t] \
                                    * beta[j][t+1] \
                                        * self.transition_matrix[i][j] \
                                            * self.emission_probability[j][peptide[t+1]]
                    
            for i in range(self.states):
                for j in range(self.states):   
                    numerator = alpha[i][t] \
                                * beta[j][t+1] \
                                    * self.transition_matrix[i][j] \
                                        * self.emission_probability[j][peptide[t+1]]     

                    xi[i, j, t] = numerator / denominator
                    # Numerator value updates with each iteration
                    # Denominator value stays the same

        return xi
    
    ###################################################################################################################
    def update_parameters(self, all_input_encodes, gamma_list, xi_list):
    
        R = len(all_input_encodes)

        final_probability = np.zeros(self.states)

        final_transition_matrix_num = np.zeros((self.states, self.states))
        final_transition_matrix_den = np.zeros(self.states)

        final_emission_probability_num = np.zeros((self.states, len(self.alphabet)))
        final_emission_probability_den = np.zeros(self.states)

        for i, r in enumerate(all_input_encodes):
            
            gamma = gamma_list[i]
            xi = xi_list[i]
            T = gamma.shape[1]

            # Initial state distribution
            final_probability += gamma[:, 0]

            # Transition matrix
            final_transition_matrix_den += np.sum(gamma[:, :T-1], axis=1)
            final_transition_matrix_num += np.sum(xi, axis=2)

            # Iterate over all the "positions" in sequence
            for t in range(T):
                seq_idx = r[t]
                a = self.alphabet[seq_idx]

                # Iterate over eveyr state
                for j in range(self.states):
                    final_emission_probability_num[j, seq_idx] += gamma[j, t]
            
            final_emission_probability_den += np.sum(gamma, axis=1)

        final_probability /= R 
        final_transition_matrix = final_transition_matrix_num / final_transition_matrix_den[:, None]
        final_emission_probability = final_emission_probability_num / final_emission_probability_den[:, None]

        return final_probability, final_transition_matrix, final_emission_probability
    
    ###################################################################################################################
    def run(self):
        peptide_nr = 0

        all_input_encodes = []
        gamma_list = []
        xi_list = []

        for peptide in self.peptides:

            peptide_nr += 1

            # Convert sequence from amino acids to numbers
            input_encode, peptide = encode(peptide, self.alphabet)
            all_input_encodes.append(input_encode)
            
            # Initialize
            alpha = self.alpha_initialize(input_encode)
            beta = self.beta_initialize(input_encode)
            
            # Calculate probabilities
            alpha, c = self.forward_probability(input_encode, alpha)
            beta = self.backward_probability(input_encode, beta, c)   

            # Expected state
            # Probability of being in state i at time t given the observed sequence Y
            gamma = self.calculate_gamma(alpha, beta)

            # Expected transition
            # Probability of being in state i and j at times t and t+1 respectively given the observed sequence Y
            xi = self.calculate_xi(input_encode, alpha, beta)

            # Store
            gamma_list.append(gamma)
            xi_list.append(xi)

        final_probability, final_transition_matrix, final_emission_probability = self.update_parameters(all_input_encodes, gamma_list, xi_list)

        # Update model
        self.initial_probability = final_probability
        self.transition_matrix = final_transition_matrix
        self.emission_probability = final_emission_probability

        return final_probability, final_transition_matrix, final_emission_probability, all_input_encodes

In [25]:
###################################################################################################################
def forward_initialize(input_encode, states, initial_probability, emission_probability):

    alpha = np.zeros(shape=(states, len(input_encode)))

    for i in range(states):
        alpha[i][0] = initial_probability[i] * emission_probability[i][input_encode[0]]

    return alpha

###################################################################################################################
def forward(input_encode, states, alpha, transition_matrix, emission_probability):
    """
    Summed probability over all paths giving rise to a given sequence
    """

    for i in range(1, len(input_encode)):

        for j in range(states):
            _sum = 0
            for k in range(states):
                _sum += emission_probability[j][input_encode[i]] * alpha[k][i-1] * transition_matrix[k][j]

            alpha[j][i] = _sum

    return alpha

###################################################################################################################
def log_likelihood(alpha):
    return np.log(np.sum(alpha[:, -1]))

In [26]:
def classify_sequence(input_encode, mito_model, cyto_model, states):

    # Define initial probability, transition matrix, and emission probability
    mito_probability, mito_transition_matrix, mito_emission_probability = mito_model
    cyto_probability, cyto_transition_matrix, cyto_emission_probability = cyto_model

    # Run forward algorithm for mito
    mito_alpha = forward_initialize(input_encode, states, mito_probability, mito_emission_probability)
    mito_alpha = forward(input_encode, states, mito_alpha, mito_transition_matrix, mito_emission_probability)
    mito_ll = log_likelihood(mito_alpha)

    # Run forward algorithm for cyto
    cyto_alpha = forward_initialize(input_encode, states, cyto_probability, cyto_emission_probability)
    cyto_alpha = forward(input_encode, states, cyto_alpha, cyto_transition_matrix, cyto_emission_probability)
    cyto_ll = log_likelihood(cyto_alpha)

    # Calculate log-likelihood ratio
    llr = mito_ll - cyto_ll

    return llr

In [27]:
def evaluate_models(test_data, mito_models, cyto_models, alphabet, n_iterations):

    accuracies = []
    llrs = []

    for iteration in range(n_iterations):
        mito_model = mito_models[iteration]
        cyto_model = cyto_models[iteration]

        correct = 0
        total = len(test_data)

        iteration_llrs = []

        for peptide, label in test_data:
            
            # Convert sequence from amino acids to numbers
            input_encode, peptide = encode(peptide, alphabet)

            # Classify test sequence
            llr = classify_sequence(input_encode, mito_model, cyto_model, states)
            iteration_llrs.append(llr)
            
            pred = 1 if llr > 0 else 0   # 1 = mito, 0 = cyto
            if pred == label:
                correct += 1

        llrs.append(iteration_llrs)
        accuracies.append(correct / total)

    return accuracies, llrs

def evaluate_model(eval_data, mito_model, cyto_model, alphabet):
    correct = 0
    for peptide, label in eval_data:
        # Convert sequence from amino acids to numbers
        input_encode, peptide = encode(peptide, alphabet)

        # Classify test sequence
        llr = classify_sequence(input_encode, mito_model, cyto_model, states)
        
        pred = 1 if llr > 0 else 0   # 1 = mito, 0 = cyto
        if pred == label:
            correct += 1

    return correct/len(eval_data), llr

In [33]:
def cross_validation(outer_data, n_iterations):

    initial_probability = [1.0/states, 1.0/states, 1.0/states, 1/ states]
    transition_matrix = np.asarray([
                                [0.80, 0.15, 0.00, 0.05],   # N‑region ('+' charged) is short
                                [0.00, 0.85, 0.10, 0.05],   # H‑region is long
                                [0.00, 0.0, 0.95, 0.05],
                                [0.3,0.2,0.2,0.9]   # C‑region has almost no transitions out
                            ])

    #n_region = np.array([0.05, 0.12, 0.04, 0.02, 0.03,   # A, R, N, D, C
    #                    0.04, 0.02, 0.05, 0.03, 0.03,   # Q, E, G, H, I
    #                    0.03, 0.12, 0.03, 0.03, 0.04,   # L, K, M, F, P
    #                    0.06, 0.06, 0.02, 0.03, 0.03    # S, T, W, Y, V
    #                    ])
    #h_region = np.array([0.03, 0.01, 0.01, 0.01, 0.02,   # A, R, N, D, C
    #                    0.02, 0.01, 0.02, 0.01, 0.12,   # Q, E, G, H, I
    #                    0.18, 0.01, 0.06, 0.10, 0.03,   # L, K, M, F, P
    #                    0.03, 0.03, 0.08, 0.03, 0.20    # S, T, W, Y, V
    #                    ])
    #c_region = np.array([0.10, 0.03, 0.04, 0.03, 0.03,   # A, R, N, D, C
    ##                    0.04, 0.03, 0.08, 0.03, 0.04,   # Q, E, G, H, I
    #                   0.04, 0.03, 0.03, 0.03, 0.04,   # L, K, M, F, P
    #                    0.12, 0.10, 0.02, 0.03, 0.03    # S, T, W, Y, V
    #                    ])

    n_region = blosum_to_emission_propabilities(BLOSUM_FILE, "K", alphabet)
    h_region = blosum_to_emission_propabilities(BLOSUM_FILE, "L", alphabet)
    c_region = blosum_to_emission_propabilities(BLOSUM_FILE, "A", alphabet)
    b_state = background_frequencies(alphabet)



    emission_probability = [n_region, h_region, c_region,b_state]

    # NOTE: to obtain 20/20/60 ratio of the original data, split has to be 0.25.
    # Split 40% into 20% test and 20% evaluate
    kf = KFold(n_splits=4, shuffle=True, random_state=42)

    all_accuracies = []
    all_llrs = []

    best_mito_models = []
    best_cyto_models = []
    for train_idx, test_idx in kf.split(outer_data):

        train_data = outer_data[train_idx]
        test_data = outer_data[test_idx]

        mito_train = [peptide for peptide, label in train_data if label == 1]
        cyto_train = [peptide for peptide, label in train_data if label == 0]

        # Initialize Baum Welch
        mito_baum = BaumWelch(mito_train, states,
                              initial_probability.copy(),
                              transition_matrix.copy(),
                              emission_probability.copy(),
                              alphabet)

        cyto_baum = BaumWelch(cyto_train, states,
                              initial_probability.copy(),
                              transition_matrix.copy(),
                              emission_probability.copy(),
                              alphabet)

        best_mito_model, best_cyto_model = None, None
        best_accuracy, best_llr = 0, 0
        llrs, accuracies = [], []
        for iteration in range(n_iterations):
            # TRAIN
            # Mito HMM and store mito model performance
            mito_probability, mito_transition_matrix, mito_emission_probability, mito_all_input_encodes = mito_baum.run()
            mito_model = (mito_probability, mito_transition_matrix, mito_emission_probability)

            # Cyto HMM and store cyto model performance
            cyto_probability, cyto_transition_matrix, cyto_emission_probability, cyto_all_input_encodes = cyto_baum.run()
            cyto_model = (cyto_probability, cyto_transition_matrix, cyto_emission_probability)

            # EVALUATE
            accuracy, llr = evaluate_model(test_data, mito_model, cyto_model, alphabet)
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_llr = llr
                best_cyto_model = cyto_model
                best_mito_model = mito_model
            llrs.append(llr)
            accuracies.append(accuracy)

        all_llrs.append(llrs)
        all_accuracies.append(accuracies)
        best_mito_models.append(best_mito_model)
        best_cyto_models.append(best_cyto_model)

    return all_accuracies, all_llrs, best_mito_models, best_cyto_models

In [34]:
states = 4

alphabet = generate_alphabet().tolist()

mito_peptides = load_peptide_data(file_name="mitochondrial_peptides")
cyto_peptides = load_peptide_data(file_name="cytosol_peptides")

# Label data to keep track of training and test portion - avoid data leakage
# Mitochondria = 1, Cytosol = 0
all_data = [(peptide, 1) for peptide in mito_peptides] + [(peptide, 0) for peptide in cyto_peptides]
all_data = np.array(all_data, dtype=object)

X = all_data[:, 0]  # First column = peptides
y = all_data[:, 1]  # Second column = labels

# 60% train, 40% test
outer_x_train, x_evaluate, outer_y_train, y_evaluate = train_test_split(X, y, test_size=0.2, random_state=42)
outer_data = np.stack((outer_x_train, outer_y_train), axis=1)

all_accuracies, all_llrs, best_mito_models, best_cyto_models = cross_validation(outer_data, n_iterations = 5)


KeyboardInterrupt: 

In [ ]:
mean_accuracy = np.mean(all_accuracies, axis=0)

plt.plot(all_accuracies)
plt.xlabel("Baum-Welch iteration")
plt.ylabel("Accuracy")
plt.title("ACCURACY change over iterations")
plt.show()